In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
SYNTHETIC DATA (VALIDATION SET) TEST
"""

import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
import jiwer
from sklearn.model_selection import train_test_split

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# ===========================
# Paths and Configurations
# ===========================
CONFIG = {
    'base_model_id': "Qwen/Qwen3-VL-2B-Instruct",
    'lora_path': "Um1neko/Qwen3VL-Instruct-2B-ManchuOCR",
    'data_parquet': "/root/autodl-fs/Manchu_OCR/train.parquet",
    'image_dir': "/root/autodl-fs/Manchu_OCR/train",
   
    'min_pixels': 32 * 32,       
    'max_pixels': 128 * 256,
   
    # Overlap tolerance threshold (%)
    'leakage_threshold': 5.0,
}

def main():
    print("\n" + "="*60)
    print("Starting Qwen3-VL Manchu OCR Evaluation (Unseen Words Mode)")
    print("="*60)
   
    # ===========================
    # 1. Data Splitting and Overlap Audit
    # ===========================
    print("\n[1/4] Reproducing dataset split and auditing overlap...")
    df = pd.read_parquet(CONFIG['data_parquet'])
   
    # Maintain the exact random seed and ratio from training to recreate the split
    train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)
   
    # Extract unique word labels from the training set
    train_vocab = set(train_df['roman'].unique())
   
    # Check how many words in the validation set appear in the training set
    overlap_mask = eval_df['roman'].isin(train_vocab)
    overlap_count = overlap_mask.sum()
    total_eval = len(eval_df)
    overlap_percentage = (overlap_count / total_eval) * 100
   
    print(f"  > Total validation samples: {total_eval}")
    print(f"  > Samples seen in training set (leakage): {overlap_count}")
    print(f"  > Label overlap rate: {overlap_percentage:.2f}%")
   
    if overlap_percentage > CONFIG['leakage_threshold']:
        print(f"\n[!] Warning: Overlap rate ({overlap_percentage:.2f}%) exceeds threshold ({CONFIG['leakage_threshold']}%).")
        print("[!] Filtering out images with seen labels...")
        # Keep only unseen words (~ acts as negation)
        eval_df = eval_df[~overlap_mask]
        print(f"  > Filtering complete. Final unseen validation set size: {len(eval_df)}")
    else:
        print(f"\n[✓] Overlap rate ({overlap_percentage:.2f}%) is within the acceptable range. Evaluating full validation set.")
   
    # Abort if the validation set is empty after filtering
    if len(eval_df) == 0:
        print("\n[x] Error: Validation set is empty after filtering. Evaluation aborted.")
        return

    # ===========================
    # 2. Load Processor and Model
    # ===========================
    print("\n[2/4] Loading base model and processor...")
    processor = AutoProcessor.from_pretrained(
        CONFIG['lora_path'], 
        min_pixels=CONFIG['min_pixels'],
        max_pixels=CONFIG['max_pixels']
    )
   
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG['base_model_id'],
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto"
    )
   
    print("\n[3/4] Integrating LoRA adapter weights...")
    model = PeftModel.from_pretrained(base_model, CONFIG['lora_path'])
    model.eval() 
    print("✓ Model and adapter loaded successfully.")

    # ===========================
    # 3. Inference Loop
    # ===========================
    print(f"\n[4/4] Starting inference on {len(eval_df)} images...")
   
    predictions = []
    references = []
   
    for index, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
        image_path = os.path.join(CONFIG['image_dir'], row['filename'])
        ground_truth = str(row['roman']).strip()
       
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            continue
           
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "识别满文单词："},
                ],
            }
        ]
       
        text_prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
       
        inputs = processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda") 
       
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, 
                max_new_tokens=30, 
                do_sample=False,   
                use_cache=True
            )
           
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
       
        pred_text = processor.batch_decode(
            generated_ids_trimmed, 
            skip_special_tokens=True, 
            clean_up_tokenization_spaces=False
        )[0].strip()
       
        predictions.append(pred_text)
        references.append(ground_truth)

    # ===========================
    # 4. Calculate Metrics (CER & WER)
    # ===========================
    print("\n" + "="*60)
    print("Evaluation complete. Calculating metrics...")
   
    clean_preds = [p if p else "<empty>" for p in predictions]
    clean_refs = [r if r else "<empty>" for r in references]
   
    cer = jiwer.cer(clean_refs, clean_preds)
    wer = jiwer.wer(clean_refs, clean_preds)
   
    print("\n[Evaluation Results (Unseen Words Mode)]")
    print(f"Valid evaluation samples: {len(clean_refs)}")
    print(f"Character Error Rate (CER): {cer * 100:.2f}%")
    print(f"Word Error Rate (WER): {wer * 100:.2f}%")
   
    print("\n[Prediction Samples (Top 5)]")
    for i in range(min(5, len(clean_refs))):
        print(f"Ground truth: {clean_refs[i]}")
        print(f"Prediction:   {clean_preds[i]}")
        print("-" * 30)

if __name__ == "__main__":
    main()


Starting Qwen3-VL Manchu OCR Evaluation (Unseen Words Mode)

[1/4] Reproducing dataset split and auditing overlap...
  > Total validation samples: 3000
  > Samples seen in training set (leakage): 1086
  > Label overlap rate: 36.20%

[!] Warning: Overlap rate (36.20%) exceeds threshold (5.0%).
[!] Filtering out images with seen labels...
  > Filtering complete. Final unseen validation set size: 1914

[2/4] Loading base model and processor...

[3/4] Integrating LoRA adapter weights...


/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


✓ Model and adapter loaded successfully.

[4/4] Starting inference on 1914 images...


Evaluating: 100%|██████████| 1914/1914 [19:40<00:00,  1.62it/s]


Evaluation complete. Calculating metrics...

[Evaluation Results (Unseen Words Mode)]
Valid evaluation samples: 1914
Character Error Rate (CER): 1.02%
Word Error Rate (WER): 6.58%

[Prediction Samples (Top 5)]
Ground truth: kurelembi
Prediction:   kurelembi
------------------------------
Ground truth: sonjome
Prediction:   sonjome
------------------------------
Ground truth: hederebumbihe
Prediction:   hederebumbihe
------------------------------
Ground truth: xeyehen
Prediction:   xeyehen
------------------------------
Ground truth: muserakvngge
Prediction:   muserakvngge
------------------------------


In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
REAL HISTORICAL DOCUMENTS TEST
"""
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import torch
import pandas as pd
import jiwer
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig
)
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# ===========================
# 1. Core Configurations
# ===========================
CONFIG = {
    'base_model_id': "Qwen/Qwen3-VL-2B-Instruct",
    'lora_path': "Um1neko/Qwen3VL-Instruct-2B-ManchuOCR",
    'test_parquet': "/root/autodl-fs/Manchu_OCR/test.parquet",
    'image_dir': "/root/autodl-fs/Manchu_OCR/test",
    'prompt_text': "识别满文单词：",
    'min_pixels': 32 * 32,
    'max_pixels': 384 * 384,
    'output_error_csv': "error_analysis_2b.csv",      # Error analysis CSV output path for 2B model
}

def main():
    print("\n" + "="*80)
    print("Starting Qwen3-VL-2B QLoRA Local Test Set Evaluation & Error Analysis")
    print("="*80)

    # --- 1. Load Data ---
    print(f"\n[1/4] Loading test set: {CONFIG['test_parquet']}")
    try:
        df = pd.read_parquet(CONFIG['test_parquet'])
        print(f"Data loaded successfully. Total test samples: {len(df)}")
    except Exception as e:
        print(f"\nError: Failed to read Parquet file: {e}")
        return

    # --- 2. Load Model ---
    print("\n[2/4] Loading 4-bit base model and LoRA weights...")
    processor = AutoProcessor.from_pretrained(
        CONFIG['lora_path'],
        min_pixels=CONFIG['min_pixels'],
        max_pixels=CONFIG['max_pixels']
    )

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG['base_model_id'],
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto",
        quantization_config=quantization_config,
    )

    model = PeftModel.from_pretrained(base_model, CONFIG['lora_path'])
    model.eval()

    # --- 3. Inference ---
    print("\n[3/4] Starting inference...")
    predictions = []
    ground_truths = []
    filenames = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
        img_path = os.path.join(CONFIG['image_dir'], row['filename'])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Warning: Skipping unreadable image: {img_path} | Error: {e}")
            continue

        target_text = str(row['roman']).strip()
        filenames.append(row['filename'])

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": CONFIG['prompt_text']},
                ],
            }
        ]

        text_prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        pred_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        predictions.append(pred_text)
        ground_truths.append(target_text)

    if len(predictions) == 0:
        print("\nError: Evaluation failed. No samples were successfully processed. Please check the image paths.")
        return

    # --- 4. Calculate Overall Metrics ---
    print("\n[4/4] Calculating overall CER/WER...")
    cer = jiwer.cer(ground_truths, predictions)
    wer = jiwer.wer(ground_truths, predictions)

    print("\n" + "="*70)
    print("Overall Evaluation Results (Qwen3-VL-2B QLoRA)")
    print("="*70)
    print(f"Valid samples             : {len(predictions)}")
    print(f"Character Error Rate (CER): {cer * 100:.4f}%")
    print(f"Word Error Rate (WER)     : {wer * 100:.4f}%")
    print("="*70)

    # ========================
    # Complete Error Analysis
    # ========================
    print("\nGenerating detailed error analysis...")

    individual_cers = [jiwer.cer([gt], [pred]) for gt, pred in zip(ground_truths, predictions)]

    error_df = pd.DataFrame({
        'filename': filenames,
        'ground_truth': ground_truths,
        'prediction': predictions,
        'cer': individual_cers
    })

    error_df.to_csv(CONFIG['output_error_csv'], index=False, encoding='utf-8')
    print(f"Error analysis CSV saved to -> {CONFIG['output_error_csv']}")

    print("\n" + "="*70)
    print("Error Analysis Statistical Summary (2B)")
    print("="*70)
    print(f"Mean CER                 : {error_df['cer'].mean()*100:.4f}%")
    print(f"Median CER               : {error_df['cer'].median()*100:.4f}%")
    print(f"Standard Deviation       : {error_df['cer'].std()*100:.4f}%")
    print(f"Worst Sample CER         : {error_df['cer'].max()*100:.4f}%")
    print(f"Best Sample CER          : {error_df['cer'].min()*100:.4f}%")
    print(f"Completely Correct Samples: {(error_df['cer'] == 0).sum()} / {len(error_df)}")
    print("="*70)

    # Top 10 worst error cases
    worst_errors = error_df.nlargest(10, 'cer').copy()
    print("\nTop 10 Worst Error Cases (Sorted by individual CER descending):")
    print("-" * 80)
    for i, row in worst_errors.iterrows():
        print(f"[{i+1:2d}] CER = {row['cer']*100:6.2f}%")
        print(f"   Ground Truth : {row['ground_truth']}")
        print(f"   Prediction   : {row['prediction']}")
        print(f"   Filename     : {row['filename']}")
        print("-" * 80)

    # Top 5 error sampling
    print("\nTop 5 Error Samples Quick Check:")
    error_count = 0
    for gt, pred in zip(ground_truths, predictions):
        if gt != pred:
            print(f"   GT  : {gt}")
            print(f"   Pred: {pred}")
            print("   " + "-"*60)
            error_count += 1
            if error_count >= 5:
                break
    if error_count == 0:
        print("No errors found in the sampled batch.")

    print("\nEvaluation and error analysis completed.")
    print(f"Complete error report saved to -> {CONFIG['output_error_csv']}")

if __name__ == "__main__":
    main()


Starting Qwen3-VL-2B QLoRA Local Test Set Evaluation & Error Analysis

[1/4] Loading test set: /root/autodl-fs/Manchu_OCR/test.parquet
Data loaded successfully. Total test samples: 218

[2/4] Loading 4-bit base model and LoRA weights...


/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)



[3/4] Starting inference...


Evaluating: 100%|██████████| 218/218 [02:08<00:00,  1.69it/s]


[4/4] Calculating overall CER/WER...

Overall Evaluation Results (Qwen3-VL-2B QLoRA)
Valid samples             : 218
Character Error Rate (CER): 22.5126%
Word Error Rate (WER)     : 59.6330%

Generating detailed error analysis...
Error analysis CSV saved to -> error_analysis_2b.csv

Error Analysis Statistical Summary (2B)
Mean CER                 : 24.8586%
Median CER               : 20.0000%
Standard Deviation       : 29.2545%
Worst Sample CER         : 150.0000%
Best Sample CER          : 0.0000%
Completely Correct Samples: 88 / 218

Top 10 Worst Error Cases (Sorted by individual CER descending):
--------------------------------------------------------------------------------
[65] CER = 150.00%
   Ground Truth : an
   Prediction   : ica
   Filename     : 00065.jpg
--------------------------------------------------------------------------------
[207] CER = 150.00%
   Ground Truth : de
   Prediction   : cin
   Filename     : 00207.jpg
--------------------------------------------------

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
QLORA TUNING FOR QWEN3VL-2B
"""
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import torch
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,  # ← 新增 QLoRA 必需
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

# ===========================
# 核心配置（QLoRA + 3090 极致优化）
# ===========================
CONFIG = {
    'model_id': "Qwen/Qwen3-VL-2B-Instruct",
    'data_parquet': "/root/autodl-fs/Manchu_OCR/train.parquet",
    'image_dir': "/root/autodl-fs/Manchu_OCR/train",
    'output_dir': "/root/autodl-tmp/qwen_lora_output_2b_qlora",
    
    # 【2B 专属大 batch 优化】
    'epochs': 9,
    'batch_size': 192,           # ← 充分利用 24GB！原 16 → 96
    'gradient_accum': 1,        # ← accum=1，每步直接吃满显存 + 最快速度
    'learning_rate': 2e-4,
    
    'min_pixels': 32 * 32,
    'max_pixels': 384 * 384,    # ← 提高分辨率，吃更多显存 + 满文识别更准
    
    # DataLoader 抗 CPU 瓶颈（大 batch 必备）
    'dataloader_num_workers': 16,
    'dataloader_pin_memory': True,
    'dataloader_persistent_workers': True,
    'dataloader_prefetch_factor': 4,
}

# ===========================
# 数据集构造（完全保留你最新的 tensor 维度修复）
# ===========================
class ManchuWordDataset(Dataset):
    def __init__(self, df, processor, image_dir):
        self.df = df
        self.processor = processor
        self.image_dir = image_dir
        self.ignore_index = -100
        self.prompt_text = "识别满文单词："
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row['filename'])
        target_text = row['roman']
       
        image = Image.open(image_path).convert("RGB")
       
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": self.prompt_text},
                ],
            },
            {
                "role": "assistant",
                "content": target_text
            }
        ]
        text_prompt = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
       
        # 【核心修复】：只 squeeze 文本张量，保留视觉特征二维结构
        for k in ["input_ids", "attention_mask"]:
            if k in inputs:
                inputs[k] = inputs[k].squeeze(0)
       
        # Loss Masking
        input_ids = inputs["input_ids"]
        labels = input_ids.clone()
       
        prompt_only_messages = [messages[0]]
        prompt_only_text = self.processor.apply_chat_template(
            prompt_only_messages, tokenize=False, add_generation_prompt=True
        )
        prompt_only_inputs = self.processor(
            text=[prompt_only_text], images=image_inputs, videos=video_inputs, return_tensors="pt"
        )
        prompt_length = prompt_only_inputs["input_ids"].squeeze(0).shape[0]
       
        labels[:prompt_length] = self.ignore_index
        inputs["labels"] = labels
       
        return inputs

def collate_fn(batch):
    # 【核心修复】：动态且安全的批次拼接
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]
   
    batch_dict = {
        "input_ids": torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0),
        "labels": torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100),
    }
    batch_dict["attention_mask"] = batch_dict["input_ids"].ne(0).long()
   
    # 动态拼接视觉特征（保留高阶维度）
    if "pixel_values" in batch[0] and batch[0]["pixel_values"] is not None:
        batch_dict["pixel_values"] = torch.cat([item["pixel_values"] for item in batch], dim=0)
   
    if "image_grid_thw" in batch[0] and batch[0]["image_grid_thw"] is not None:
        batch_dict["image_grid_thw"] = torch.cat([item["image_grid_thw"] for item in batch], dim=0)
    return batch_dict

# ===========================
# 主流程
# ===========================
def main():
    print("\n" + "="*80)
    print("Qwen3-VL-2B QLoRA 4bit 加速版启动 (单 3090 充分利用 24GB 显存)")
    print("batch=96 + accum=1 + max_pixels=384*384 → 预计峰值 21~23GB")
    print("="*80)
   
    print(f"\n[1/4] 加载 Processor 并约束像素...")
    processor = AutoProcessor.from_pretrained(
        CONFIG['model_id'],
        min_pixels=CONFIG['min_pixels'],
        max_pixels=CONFIG['max_pixels']
    )
   
    print(f"[2/4] 加载数据集: {CONFIG['data_parquet']} ...")
    df = pd.read_parquet(CONFIG['data_parquet'])
    train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)
   
    train_dataset = ManchuWordDataset(train_df, processor, CONFIG['image_dir'])
    eval_dataset = ManchuWordDataset(eval_df, processor, CONFIG['image_dir'])
    print(f"训练集: {len(train_dataset)} 条, 验证集: {len(eval_dataset)} 条")
    
    print("\n[3/4] 加载模型 + 4bit 量化 (QLoRA)...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,   # 再省一点显存
        bnb_4bit_quant_type="nf4"         # 最优量化方式
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG['model_id'],
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto",
        quantization_config=quantization_config,   # ← QLoRA 核心
    )
    model.gradient_checkpointing_enable()
    
    print("\n配置 LoRA...")
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=64,
        lora_alpha=128,
        lora_dropout=0.05,
        target_modules="all-linear",
        modules_to_save=["embed_tokens", "lm_head"]
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    
    # 显存监控（关键！）
    print(f"模型+LoRA加载后显存占用: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"峰值已分配显存: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    
    print("\n[4/4] 启动训练... (batch=96 + accum=1)")
    training_args = TrainingArguments(
        output_dir=CONFIG['output_dir'],
        per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accum'],
        learning_rate=CONFIG['learning_rate'],
        num_train_epochs=CONFIG['epochs'],
        bf16=True,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=3,
        remove_unused_columns=False,
        report_to="none",
        
        # === DataLoader 极致优化 ===
        dataloader_num_workers=CONFIG['dataloader_num_workers'],
        dataloader_pin_memory=CONFIG['dataloader_pin_memory'],
        dataloader_persistent_workers=CONFIG['dataloader_persistent_workers'],
        dataloader_prefetch_factor=CONFIG['dataloader_prefetch_factor'],
        
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
    )
    trainer.train()
   
    print("\n保存最终模型...")
    final_save_path = os.path.join(CONFIG['output_dir'], "final_lora")
    trainer.model.save_pretrained(final_save_path)
    processor.save_pretrained(final_save_path)
    print(f"✓ 训练完成！QLoRA 权重已保存至: {final_save_path}")

if __name__ == "__main__":
    main()


Qwen3-VL-2B QLoRA 4bit 加速版启动 (单 3090 充分利用 24GB 显存)
batch=96 + accum=1 + max_pixels=384*384 → 预计峰值 21~23GB

[1/4] 加载 Processor 并约束像素...


`torch_dtype` is deprecated! Use `dtype` instead!


[2/4] 加载数据集: /root/autodl-fs/Manchu_OCR/train.parquet ...
训练集: 57000 条, 验证集: 3000 条

[3/4] 加载模型 + 4bit 量化 (QLoRA)...

配置 LoRA...


/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 720,896,000 || all params: 2,848,428,032 || trainable%: 25.3086
模型+LoRA加载后显存占用: 3.00 GB
峰值已分配显存: 3.00 GB

[4/4] 启动训练... (batch=96 + accum=1)


/root/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.454200,0.341226
2,0.182700,0.181555
3,0.101200,0.124238
4,0.050000,0.107034
5,0.019100,0.091741
6,0.005000,0.081778
7,0.001300,0.080009
8,0.000900,0.079605
9,0.000800,0.079344


/root/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



保存最终模型...
✓ 训练完成！QLoRA 权重已保存至: /root/autodl-tmp/qwen_lora_output_2b_qlora/final_lora
